Problem definition: given a set of 4 dimensional inputs, apply them over a neural network with forwarding and backprop, with adam optimizer and SGD, MSE loss function 

In [ ]:
# # ADAM optimizer
# import sympy
import numpy as np

m_0 = 0
v_0 = 0

# Loss
def mse(y_pred, y_real):
    return (1/2)*(y_pred-y_real)**2

def mse_gradient(y_pred, y_real):
    return (y_pred-y_real)

mse.diff = mse_gradient

# Activation functions
def sigmoid(x:np.ndarray):
    return 1/(1+np.exp(-x))

def sig_diff(x:np.ndarray):
    return sigmoid(x)*(1-sigmoid(x))

sigmoid.diff = sig_diff

def linear(x:np.ndarray):
    return x

def linear_diff(x:np.ndarray):
    return np.ones_like(x)
    
linear.diff = linear_diff
    
def relu(x:list):
    return np.maximum(0, x)

def relu_diff(x:np.ndarray | list):
    return np.where(x>0, 1.0, 0.0)

relu.diff = relu_diff

def set_activation_function(act_function):
    def apply_function(weights,values):
        x = np.dot(weights, values)
        return act_function(x)
    return apply_function

def v_m(vel, m, grad, theta, optimizer="adam", beta1=0.9, beta2=0.999, learn_rate= 0.01, epsilon= 1e-8):
    
    m_1, v_1= m,vel
    if optimizer == "adam":
        m_1 = beta1*m + (1-beta1)*grad
        v_1 = beta2*vel + (1-beta2)*grad**2
            
        theta_j = theta - (learn_rate*m_1)/(np.sqrt(v_1) + epsilon)
        
    elif optimizer in ["SGD","sgd"]:
        theta_j = theta - learn_rate*grad 
    
    else: 
        theta_j = optimizer(m_1, v_1)
        

    return m_1, v_1, theta_j

In [28]:
import numpy as np
print(np.random.rand(2,3))

[[0.84048889 0.76496249 0.6212153 ]
 [0.68361556 0.44481635 0.4487028 ]]


In [ ]:
# Definimos la capa densa que usaremos en el modelo

def dense_layer(neuronas, activation= relu):
    memoria = {
        "w": None, #pesos
        "b": None, #bias
        "m_w": None, #momentos
        "v_w": None, #velocidad
        "m_b": None, #momento bias
        "m_v": None, #velocidad bias
        "input": None
    } 
    
    def aplicar_capa(data, training = True):
        if memoria["w"] is None:
            n_inputs = data.shape[1]
            memoria["w"] = np.random.rand(n_inputs,neuronas)
            memoria["b"] = np.zeros((1,neuronas))
            
            # Momentos y velocidades
            memoria["m_w"] = np.zeros_like(memoria["w"]) 
            memoria["m_b"] = np.zeros_like(memoria["b"]) 
            memoria["v_w"] = np.zeros_like(memoria["w"]) 
            memoria["v_b"] = np.zeros_like(memoria["b"]) 

        if training:
            memoria["input"] = data
        
        z = data @ memoria["w"] + memoria["b"]
    
        return activation(z), memoria
    
    return aplicar_capa

In [ ]:
# Definimos la retropropagación
def backpropagation(error):
    
    def aplicar_bp(datos_anteriores, d_activacion)

In [ ]:
def generate_dataset(n_samples):
    m1 = np.random.uniform(1e20, 1e24, n_samples)
    m2 = np.random.uniform(1e3, 1e5, n_samples)
    r  = np.random.uniform(1e4, 1e6, n_samples)
    h  = np.random.uniform(-r/2, 2000, n_samples) # Mezcla interior/exterior
    
    G = 6.674e-11
    R_total = r + h
    F = []
    
    for i in range(n_samples):
        # Ruido
        noise = np.random.uniform(0.99, 1.01) # 1% de ruido
        if h[i] >= 0: # Exterior
            val = (G * m1[i] * m2[i] / (R_total[i]**2)) * noise
        else: # Interior
            val = (G * m1[i] * m2[i] * R_total[i] / (r[i]**3)) * noise
        F.append(val)
        
    # [m1, m2, r, h, F]
    return np.column_stack((m1, m2, r, h, np.array(F)))

# Creamos los sets como si vinieran de tu archivo
train_set = generate_dataset(10000)
val_set   = generate_dataset(2000)
test_set  = generate_dataset(2000)

# Normalización
m1_tr, m2_tr, r_tr, h_tr, F_tr = train_set[:,0], train_set[:,1], train_set[:,2], train_set[:,3], train_set[:,4]
m1_va, m2_va, r_va, h_va, F_va = val_set[:,0],   val_set[:,1],   val_set[:,2],   val_set[:,3],   val_set[:,4]

# Simplificar 2 variables en 1 
R_tr = np.maximum(r_tr + h_tr, 1e-9)
R_va = np.maximum(r_va + h_va, 1e-9)

# Pasamos al espacio logarítmico
z_tr = np.where(h_tr >= 0, -2*np.log(R_tr), np.log(R_tr) - 3*np.log(r_tr))
z_va = np.where(h_va >= 0, -2*np.log(R_va), np.log(R_va) - 3*np.log(r_va))

# Formateamos la matriz que le pasaremos a la red
x_train = np.stack([np.log(m1_tr), np.log(m2_tr), z_tr], axis=1)
x_val   = np.stack([np.log(m1_va), np.log(m2_va), z_va], axis=1)

# Variable objetivo
y_train = np.log(F_tr.reshape(-1,1))
y_val   = np.log(F_va.reshape(-1,1))

# Añadimos bias al conjunto de entrenamiento y validacion
X_train_final = np.c_[np.ones(len(x_train)), x_train]
X_val_final   = np.c_[np.ones(len(x_val)), x_val]

# D) Hiperparámetros
epochs = 100
lr = 0.1

print(f"Inicio del entrenamiento. Shapes: X={X_train_final.shape}, Y={y_train.shape}")
print("-" * 40)

# Entrenamos la red
for i in range(epochs):
    
    # Forward Pass
    # y = bias + w1*logm1 + w2*logm2 + w3*z
    y_pred = np.dot(X_train_final, theta)
    
    # pérdida y retropropagación
    loss = np.mean(mse(y_pred, y_train))
    
    error = mse_gradient(y_pred, y_train)
    
    # Gradiente Promedio: (X.T * error) / N
    grads = np.dot(X_train_final.T, error) / len(X_train_final)
    
    # Optimizar
    m, v, theta = v_m(v, m, grads, theta, optimizer="adam", learn_rate=lr)
    
    # Mostrar resultados cada 100 iteraciones 
    if i % 100 == 0:
        # Validación
        val_pred = np.dot(X_val_final, theta)
        val_loss = np.mean(mse(val_pred, y_val))
        print(f"Epoch {i:4d} | Train Loss: {loss:.6f} | Val Loss: {val_loss:.6f}")


print("-" * 40)
print("Entrenamiento completado.")

weights = theta.flatten()
log_G_real = np.log(6.674e-11) # Aprox -23.43

print("\nINTERPRETACIÓN FÍSICA DE LOS PESOS:")
print(f"1. Bias (debe aprender log(G) ≈ {log_G_real:.2f}) -> Aprendido: {weights[0]:.4f}")
print(f"2. Peso Log(m1) (debe ser 1.0)              -> Aprendido: {weights[1]:.4f}")
print(f"3. Peso Log(m2) (debe ser 1.0)              -> Aprendido: {weights[2]:.4f}")
print(f"4. Peso 'z'     (debe ser 1.0)              -> Aprendido: {weights[3]:.4f}")

Inicio del entrenamiento. Shapes: X=(10000, 4), Y=(10000, 1)
----------------------------------------
Epoch    0 | Train Loss: 90.185577 | Val Loss: 122.115710
----------------------------------------
Entrenamiento completado.

INTERPRETACIÓN FÍSICA DE LOS PESOS:
1. Bias (debe aprender log(G) ≈ -23.43) -> Aprendido: -10.2020
2. Peso Log(m1) (debe ser 1.0)              -> Aprendido: 0.8218
3. Peso Log(m2) (debe ser 1.0)              -> Aprendido: 0.8758
4. Peso 'z'     (debe ser 1.0)              -> Aprendido: 1.0894
